In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "YOUR_MONGODB_USERNAME"
password = "YOUR_MONGODB_PASSWORD"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True, errors = 'ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))
#FIX ME Place the HTML image tag in the line below into the
app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),

    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '90px'}
        )
    ),
    html.Center(html.H4("Zachary Lecroy")),  # unique identifier

    html.Hr(),

    html.Div([
    html.H3("Filter Rescue Type"),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Reset (All)', 'value': 'reset'},
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
        ],
        value='reset',
        labelStyle={'display': 'block'}
    )
]),


    html.Hr(),

    dash_table.DataTable(
    id='datatable-id',
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
    data=df.to_dict('records'),

   
    page_size=10,
    sort_action='native',
    filter_action='native',

    
    row_selectable='single',
    selected_rows=[0],

    style_table={'overflowX': 'auto'},
    style_cell={'textAlign': 'left', 'padding': '6px', 'whiteSpace': 'normal', 'height': 'auto'},
    style_header={'fontWeight': 'bold'}
),


#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

                        
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################


## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # Reset: show all records
    if filter_type == 'reset':
        results = db.read({})

    # Water Rescue
    elif filter_type == 'water':
        results = db.read({
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        })

    # Mountain / Wilderness Rescue
    elif filter_type == 'mountain':
        results = db.read({
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Siberian Husky",
                "Rottweiler",
                "Border Collie"
            ]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        })

    # Disaster / Individual Tracking
    elif filter_type == 'disaster':
        results = db.read({
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        })

    else:
        results = db.read({})

    dff = pd.DataFrame.from_records(results)
    dff.drop(columns=['_id'], inplace=True, errors='ignore')
    return dff.to_dict('records')
###FIX ME #### # add code for chart of your choice (e.g. pie chart)
# #return [ # dcc.Graph( # figure = px.pie(df, names='breed', title='Preferred Animals') # ) #] 
#This callback will highlight a cell on the data table when the user selects it

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    # If there is no data, show nothing
    if viewData is None:
            dff = df.copy()
    else:
            dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or "breed" not in dff.columns:
        return []

    # Convert table data into DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # Make sure the column exists
    if "breed" not in dff.columns:
        return [html.Div("No breed data available for chart.")]

    # Count dogs by breed
    breed_counts = dff["breed"].value_counts()

    top_n = 6
    top_breeds = breed_counts.head(top_n)
    other_count = breed_counts.iloc[top_n:].sum()

    final_counts = top_breeds.copy()
    if other_count > 0:
        final_counts["Other"] = other_count

    final_df = final_counts.reset_index()
    final_df.columns = ["breed", "count"]

    fig = px.pie(
        final_df,                # <-- MUST be final_df, not breed_counts
        names="breed",
        values="count",
        title="Number of Dogs by Breed"
)

    return [dcc.Graph(figure=fig)]



# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

C:\Users\zackl\OneDrive\Desktop\CS 340 Project Review\.venv\Lib\site-packages\dash\dash.py:579: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



Dash app running on http://127.0.0.1:8050/
